## Import Library and Packages

In [73]:
pip install pandas numpy scikit-learn xgboost sentence-transformers nltk --quite


Usage:   
  pip3 install [options] <requirement specifier> [package-index-options] ...
  pip3 install [options] -r <requirements file> [package-index-options] ...
  pip3 install [options] [-e] <vcs project url> ...
  pip3 install [options] [-e] <local project path> ...
  pip3 install [options] <archive url/path> ...

no such option: --quite


In [74]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score
from xgboost import XGBClassifier
from sentence_transformers import SentenceTransformer

plt.style.use("seaborn-v0_8")

## Import Dataset

In [75]:
df = pd.read_csv("new_confident_dataset.csv")

In [76]:
df["confidence_level"] = df["confidence_level"].str.strip().str.lower()


In [77]:
df["confidence_label"] = df["confidence_level"].map({
    "low": 0,
    "medium": 0,  # Treating medium as low confidence
    "high": 1
})


In [78]:
df = df.dropna(subset=["confidence_label"])
df["confidence_label"] = df["confidence_label"].astype(int)

In [79]:
print("Final class distribution:")
print(df["confidence_label"].value_counts())

Final class distribution:
confidence_label
1    6777
0    3262
Name: count, dtype: int64


In [80]:
def clean_text(text):
    text = str(text).lower().strip()
    text = re.sub(r"\s+", " ", text)
    return text

df["clean_text"] = df["transcription"].apply(clean_text)

## Feature Extraction

In [81]:
HEDGE_WORDS = {"maybe", "perhaps", "probably", "i think", "i guess", "not sure", "kind of", "sort of"}
CERTAINTY_WORDS = {"definitely", "sure", "certainly", "i know", "absolutely", "clearly", "obviously"}
FILLERS = {"uh", "um", "like", "you know"}

def extract_linguistic_features(text):
    words = text.split()
    word_count = len(words)

    features = {
        "hedge_count": sum(1 for w in HEDGE_WORDS if w in text),
        "certainty_count": sum(1 for w in CERTAINTY_WORDS if w in text),
        "question_count": text.count("?"),
        "negation_count": sum(1 for w in ["not", "no", "never", "don't", "didn't", "can't"] if w in words),
        "modal_count": sum(1 for w in ["might", "could", "should", "would", "may"] if w in words),
        "filler_count": sum(1 for w in FILLERS if w in text),
        "avg_word_length": np.mean([len(w) for w in words]) if word_count else 0,
        "sentence_length": word_count,
        "first_person_ratio": words.count("i") / word_count if word_count else 0,
    }
    return features

In [82]:
ling_feauters = df["clean_text"].apply(extract_linguistic_features)
ling_df = pd.DataFrame(list(ling_feauters))

## NLP Model

In [83]:
embedder = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = embedder.encode(
    df["clean_text"].tolist(),
    show_progress_bar=True,
    batch_size=32
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/314 [00:00<?, ?it/s]

In [84]:
print(f"Embeddings shape: {embeddings.shape}")

# Combine features
X = np.hstack([
    embeddings,
    ling_df.values
])

Embeddings shape: (10039, 384)


In [85]:
y = df["confidence_label"].values

print(f"\nFinal feature matrix shape: {X.shape}")
print(f"Target distribution: {np.bincount(y)}")


Final feature matrix shape: (10039, 393)
Target distribution: [3262 6777]


## Train, Test, Validation Dataset Split

In [86]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42
)

In [87]:
print(f"Train set: {X_train.shape}, class distribution: {np.bincount(y_train)}")
print(f"Val set: {X_val.shape}, class distribution: {np.bincount(y_val)}")
print(f"Test set: {X_test.shape}, class distribution: {np.bincount(y_test)}")

Train set: (7027, 393), class distribution: [2283 4744]
Val set: (1506, 393), class distribution: [ 490 1016]
Test set: (1506, 393), class distribution: [ 489 1017]


### Applying Scaler

In [88]:
scaler = StandardScaler()

num_ling_features = ling_df.shape[1]
X_train[:, -num_ling_features:] = scaler.fit_transform(X_train[:, -num_ling_features:])
X_val[:, -num_ling_features:] = scaler.transform(X_val[:, -num_ling_features:])
X_test[:, -num_ling_features:] = scaler.transform(X_test[:, -num_ling_features:])

## Logistic Regression Model

In [89]:
logreg = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)
logreg.fit(X_train, y_train)

LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)

## XGB Model

In [90]:
xgb = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42
)

xgb.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.05, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=4, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=300, n_jobs=None,
              num_parallel_tree=None, ...)

In [91]:
def evaluate(model, X, y, dataset_name=""):
    print(f"\n{dataset_name} Results:")
    print("-" * 40)

    probs = model.predict_proba(X)[:, 1]
    preds = (probs >= 0.5).astype(int)

    print("\nConfusion Matrix:")
    print(confusion_matrix(y, preds))

    print("\nClassification Report:")
    print(classification_report(y, preds, target_names=["Low Confidence", "High Confidence"]))

    print(f"ROC-AUC Score: {roc_auc_score(y, probs):.4f}")
    print(f"Accuracy: {accuracy_score(y, preds):.4f}")


## Evaluation for Logistic Regression

In [92]:
evaluate(logreg, X_test, y_test, "Test Set")


Test Set Results:
----------------------------------------

Confusion Matrix:
[[421  68]
 [125 892]]

Classification Report:
                 precision    recall  f1-score   support

 Low Confidence       0.77      0.86      0.81       489
High Confidence       0.93      0.88      0.90      1017

       accuracy                           0.87      1506
      macro avg       0.85      0.87      0.86      1506
   weighted avg       0.88      0.87      0.87      1506

ROC-AUC Score: 0.9486
Accuracy: 0.8718


## Evaluation for XGB

In [93]:
evaluate(xgb, X_test, y_test, "Test Set")


Test Set Results:
----------------------------------------

Confusion Matrix:
[[380 109]
 [ 66 951]]

Classification Report:
                 precision    recall  f1-score   support

 Low Confidence       0.85      0.78      0.81       489
High Confidence       0.90      0.94      0.92      1017

       accuracy                           0.88      1506
      macro avg       0.87      0.86      0.86      1506
   weighted avg       0.88      0.88      0.88      1506

ROC-AUC Score: 0.9580
Accuracy: 0.8838


In [94]:
def predict_confidence(text, model, threshold=0.5):
    """Predict confidence level for a given text."""
    text = clean_text(text)

    # Generate embedding
    emb = embedder.encode([text])

    # Extract linguistic features
    ling = extract_linguistic_features(text)
    ling_vec = np.array(list(ling.values())).reshape(1, -1)
    ling_vec_scaled = scaler.transform(ling_vec)

    # Combine features
    X_input = np.hstack([emb, ling_vec_scaled])

    # Predict
    prob = model.predict_proba(X_input)[0][1]
    label = "high" if prob >= threshold else "low"

    return {
        "confidence_label": label,
        "confidence_score": float(prob),
        "linguistic_features": ling
    }


## Testing Samples

In [95]:
test_texts = [
    "I'm absolutely certain this is the right approach.",
    "Um, maybe we could try that, I guess?",
    "I don't know, perhaps it might work.",
    "uh, i guess this is not working",
      "This is definitely the best solution."
]

print("\nUsing XGBoost model for predictions:\n")
for text in test_texts:
    result = predict_confidence(text, xgb)
    print(f"Text: '{text}'")
    print(f"Prediction: {result['confidence_label']} (score: {result['confidence_score']:.3f})")
    print()




Using XGBoost model for predictions:

Text: 'I'm absolutely certain this is the right approach.'
Prediction: high (score: 0.953)

Text: 'Um, maybe we could try that, I guess?'
Prediction: low (score: 0.352)

Text: 'I don't know, perhaps it might work.'
Prediction: low (score: 0.326)

Text: 'uh, i guess this is not working'
Prediction: low (score: 0.487)

Text: 'This is definitely the best solution.'
Prediction: high (score: 0.984)

